# Keras vs PyTorch: Image Classification Comparison

This notebook compares two popular deep learning frameworks used for image classification:

- Keras (high-level API built on TensorFlow)
- PyTorch (more flexible and Pythonic deep learning framework)

Both frameworks can build the same type of CNN classifier, but the syntax and training flow are different.

In this notebook, we explain the main differences and show a simplified example of each approach.

## 1. Key differences

### Keras
- easier to write and read
- good for quick prototyping
- uses `model.fit()` for training
- simple model definition with `Sequential`

### PyTorch
- more control over training steps
- explicit forward and backward pass
- uses `optimizer.step()` and `loss.backward()`
- more flexible for research and custom models

Both are powerful, and the choice depends on the project and the developer preference.

## 2. Keras example

This is the Keras version of a simple CNN classifier.

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATASET_PATH = os.path.join('.', 'images_dataSAT')

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_generator, validation_data=val_generator, epochs=3, verbose=0)
print('Keras training finished')

## 3. PyTorch example

This is the PyTorch version of the same idea. It uses a dataset loader and a training loop.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATASET_PATH = os.path.join('.', 'images_dataSAT')

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    for images, labels in train_loader:
        labels = labels.float().view(-1, 1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

print('PyTorch training finished')

## 4. Side-by-side summary

| Aspect | Keras | PyTorch |
|---|---|---|
| Model creation | Simple and compact | More explicit and flexible |
| Training | `model.fit()` | manual training loop |
| Best for | quick experiments | custom research and control |
| Code style | high-level | low-level but powerful |

Both frameworks are widely used in deep learning and can solve the same image classification task.

## 5. Conclusion

Keras is usually easier for beginners because it reduces boilerplate code and makes the workflow simpler.

PyTorch is often preferred for detailed control, custom training logic, and research projects.

The choice depends on your learning goal, project requirement, and personal preference.